# FIP B9 - Real-Time Information Assistant

**LangChain tools over two free public APIs**

- Weather: Open-Meteo geocoding + current forecast
- Currency: Frankfurter v2 reference rates
- Routing: `ChatOpenAI` live, `FakeListChatModel` offline

> The offline cells use **simulated LLM routing** and are labelled as simulations.
> No OpenAI API call is made in this notebook by default.
> Frankfurter rates are reference rates, not live trading quotes.

## 1. Objectives

- Call a geocoding API and a weather API safely.
- Call an exchange-rate API, validate inputs, and convert amounts.
- Wrap both integrations as LangChain tools.
- Let a chat model choose the correct tool from a plain-language question.
- Test everything offline with mocks and `FakeListChatModel`.
- Keep the live OpenAI path optional and key-safe.

## 2. Setup

Run the notebook from the project root so `src` is importable. Dependencies are
installed from `requirements.txt`:

```powershell
.\.venv\Scripts\python.exe -m pip install -r requirements.txt
```

No OpenAI key is needed for the weather and currency API demos. A key is needed
only for the live routing cell in Section 8.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").is_dir())

## 3. Architecture

```text
question
   |
   v
chat model (live ChatOpenAI or FakeListChatModel)
   |
   v
validated JSON: {"tool": ..., "arguments": {...}}
   |
   +--> current_weather  --> Open-Meteo
   +--> exchange_rate    --> Frankfurter
   +--> convert_currency --> Frankfurter + local arithmetic
   |
   v
readable answer
```

The model only selects the tool. Weather and rate values always come from the
public APIs, never from the model.

## 4. Weather client (Open-Meteo)

`geocode_city()` resolves a name to coordinates, then `get_current_weather()`
requests current conditions from the forecast endpoint. Unknown cities raise
`LocationNotFoundError`; timeouts and HTTP errors raise project exceptions.

The cell below needs internet access. It makes **no OpenAI call**.

In [ ]:
from src.weather_api import geocode_city, get_current_weather

location = geocode_city("Berlin")
print(f"{location.label()} -> {location.latitude:.4f}, {location.longitude:.4f}")

weather = get_current_weather("Berlin")
print(weather.summary())

### Weather codes

Open-Meteo returns WMO weather codes. The project maps them to descriptions and
shows `Unknown weather code (n)` instead of guessing.

In [ ]:
from src.weather_api import describe_weather_code

for code_value in (0, 3, 61, 95, 999):
    print(code_value, "->", describe_weather_code(code_value))

## 5. Currency client (Frankfurter v2)

Frankfurter returns an array of reference-rate objects:

```json
[{"date": "2026-11-16", "base": "USD", "quote": "EUR", "rate": 0.90573}]
```

Rates are published by the European Central Bank for reference. They are **not
live trading quotes** and are not suitable for trading or settlement.

Inputs are validated before any request: three-letter codes only, positive
finite amounts, and a local shortcut when both currencies are the same.

In [ ]:
from src.currency_api import REFERENCE_RATE_NOTE, convert_currency, get_exchange_rate

rate = get_exchange_rate("USD", "EUR")
print(rate.summary())

conversion = convert_currency(100, "USD", "EUR")
print(conversion.summary())
print(REFERENCE_RATE_NOTE)

## 6. LangChain tools

Three `@tool` functions expose the clients to LangChain. Each has a Pydantic
input schema and returns readable text. Expected client errors become short
messages such as `Weather lookup failed: ...` instead of tracebacks.

In [ ]:
from src.tools import ALL_TOOLS

for item in ALL_TOOLS:
    print(f"{item.name}: {item.description.splitlines()[0]}")
    print("  args:", sorted(item.args))

## 7. Offline routing with `FakeListChatModel`

`FakeListChatModel` returns canned strings. Here the canned string is the routing
JSON that a live model would normally produce. This exercises the same
validation and tool-execution path without an OpenAI key.

**These are simulated decisions, not live OpenAI output.**

In [ ]:
from langchain_core.language_models import FakeListChatModel

from src.assistant import answer_question

llm = FakeListChatModel(
    responses=['{"tool": "current_weather", "arguments": {"city": "Tokyo"}}']
)
answer = answer_question(
    "Do I need an umbrella in Tokyo? What is the weather there?",
    llm,
    simulated=True,
)
print(answer.render())

In [ ]:
llm = FakeListChatModel(
    responses=[
        '{"tool": "convert_currency", "arguments": '
        '{"amount": 250, "from_currency": "EUR", "to_currency": "INR"}}'
    ]
)
answer = answer_question("Convert 250 EUR to INR.", llm, simulated=True)
print(answer.render())

### Why the label matters

Every simulated answer is prefixed with:

```text
[SIMULATION - FakeListChatModel routing, not a live OpenAI response]
```

Do not remove that label when sharing results, and never present these outputs as
live OpenAI responses.

## 8. Live routing (optional, needs a key)

This cell is not run by default. It raises `MissingAPIKeyError` before creating a
client if `OPENAI_API_KEY` is missing or blank.

Set `OPENAI_API_KEY` in `.env` or the environment first. Never paste a key into a
notebook cell.

In [ ]:
from src.assistant import create_chat_model
from src.errors import MissingAPIKeyError

try:
    live_llm = create_chat_model()
    print("Live model ready:", live_llm.model_name)
except MissingAPIKeyError as exc:
    print("Live mode unavailable:", exc)

## 9. Error handling

The clients distinguish unknown locations, unsupported currencies, invalid
amounts, timeouts, HTTP failures, and malformed JSON. The next cell demonstrates
local validation, which makes no network request.

In [ ]:
from src.currency_api import validate_amount
from src.errors import InputValidationError, InvalidAmountError

for value in (0, -5, "abc"):
    try:
        validate_amount(value)
    except InvalidAmountError as exc:
        print("invalid amount", value, "->", exc)

try:
    get_exchange_rate("US", "EUR")
except InputValidationError as exc:
    print("invalid code ->", exc)

## 10. Tests

Offline suite (no network, no OpenAI key):

```powershell
.\.venv\Scripts\python.exe -m unittest discover -s tests -v
```

Expected: `Ran 72 tests` with `OK (skipped=6)`.

Live public API checks (network, still no OpenAI key):

```powershell
$env:RUN_LIVE_TESTS="1"
.\.venv\Scripts\python.exe -m unittest tests.test_live_apis -v
```

Expected: 6 tests pass when Open-Meteo and Frankfurter are reachable.

## 11. Five Q&A examples

The evidence file `outputs/five_qa_evidence.md` contains five questions covering
both APIs:

| # | Question | Tool |
|---|---|---|
| 1 | What is the weather in Berlin right now? | `current_weather` |
| 2 | Do I need an umbrella in Tokyo? | `current_weather` |
| 3 | How much is 100 USD in EUR? | `convert_currency` |
| 4 | What is the exchange rate from GBP to JPY? | `exchange_rate` |
| 5 | Convert 250 EUR to INR. | `convert_currency` |

The routing is simulated with `FakeListChatModel`; the weather and currency values
come from the live free public APIs at generation time. They are **not** live
OpenAI responses.

## 12. Conclusion and limitations

- The public API integrations are complete and were verified live.
- The tool-routing path is verified offline with `FakeListChatModel`.
- Live OpenAI routing has **not** been tested: no key or API credits were available.
- Open-Meteo data is an estimate, and city names can be ambiguous.
- Frankfurter rates are reference rates, not trading quotes.
- The project has no caching, history, monitoring, or production hardening.

See `docs/risks_and_limitations.md` for the full risk discussion.